# 深度学习激活函数详解（PyTorch版）

> 本笔记本是 [ReLU_Activation_functions.ipynb](./ReLU_Activation_functions.ipynb) 的 **PyTorch 等价版本**，
> 原版使用 TensorFlow/Keras，本版使用 PyTorch 实现相同功能。

## 激活函数的作用

激活函数为神经网络引入非线性特性。没有激活函数，无论网络多深，都等价于单层线性变换。

## 激活函数演进历史

```
sigmoid/tanh → ReLU → Leaky ReLU → PReLU → ELU → SELU → Swish/GELU
```

## 主要问题与解决方案

| 问题 | 影响 | 解决方案 |
|------|------|----------|
| 梯度饱和 | sigmoid/tanh 在极值区梯度趋近于0 | 使用 ReLU 系列 |
| Dead ReLU | 负输入永久置零，神经元"死亡" | Leaky ReLU, PReLU, ELU |
| 输出不对称 | ReLU 输出均值不为0，影响下一层 | ELU, SELU |
| 计算效率 | 某些激活函数计算代价高 | ReLU 简单高效 |

## 学习目标

1. 理解为什么需要激活函数（梯度消失问题）
2. 掌握 PyTorch 中常用激活函数的使用方法
3. 理解各激活函数的数学原理与优缺点
4. 学会可视化激活函数及其导数
5. 了解不同激活函数的适用场景与选择策略

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 设置随机种子 / Set random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# 检测并选择计算设备 / Detect and select compute device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 设置中文字体 / Set Chinese font
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print(f"PyTorch版本: {torch.__version__}")
print(f"计算设备: {device}")

# 检查各激活函数在 nn 模块中的可用性 / Check activation availability in nn module
activation_classes = ['ReLU', 'LeakyReLU', 'ELU', 'SELU', 'PReLU', 'GELU', 'SiLU']
for name in activation_classes:
    cls = getattr(nn, name, None)
    if cls is not None:
        print(f"  nn.{name}: \u2713 可用 (available)")
    else:
        print(f"  nn.{name}: \u2717 不可用 (not available)")

## 1. 为什么需要激活函数 — 梯度消失问题

假设一个5层网络，每层使用 Sigmoid 激活函数。Sigmoid 导数最大值为 0.25，
5 层之后梯度最多变为 $0.25^5 \approx 0.001$，几乎消失。这就是**梯度消失问题**。

ReLU 在正区间导数恒为1，梯度不会逐层衰减，因此有效缓解了梯度消失。

In [ ]:
def sigmoid(x):
    """Sigmoid 激活函数 / Sigmoid activation function."""
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    """Sigmoid 导数 / Sigmoid derivative."""
    s = sigmoid(x)
    return s * (1 - s)

def relu_derivative(x):
    """ReLU 导数 / ReLU derivative."""
    return np.where(x > 0, 1, 0)

# 模拟梯度消失 / Simulate vanishing gradient
x = np.linspace(-6, 6, 200)

# 计算 Sigmoid 每层梯度乘积 / Compute gradient product per layer for Sigmoid
num_layers = 5
sigmoid_grad_product = sigmoid_derivative(x) ** num_layers
relu_grad_product = relu_derivative(x) ** num_layers

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sigmoid 梯度 / Sigmoid gradient
axes[0].plot(x, sigmoid_derivative(x), 'b-', linewidth=2, label='单层 Sigmoid 导数')
axes[0].plot(x, sigmoid_grad_product, 'r--', linewidth=2, label=f'{num_layers}层梯度乘积')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].set_title('Sigmoid: 梯度消失演示', fontsize=13)
axes[0].set_xlabel('x')
axes[0].set_ylabel('梯度大小')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ReLU 梯度 / ReLU gradient
axes[1].plot(x, relu_derivative(x), 'b-', linewidth=2, label='单层 ReLU 导数')
axes[1].plot(x, relu_grad_product, 'r--', linewidth=2, label=f'{num_layers}层梯度乘积')
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_title('ReLU: 正区间梯度不衰减', fontsize=13)
axes[1].set_xlabel('x')
axes[1].set_ylabel('梯度大小')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('为什么需要激活函数：梯度消失问题', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n关键发现：Sigmoid 在5层后梯度几乎为0，而 ReLU 在正区间梯度始终为1，不会消失。")
print("这就是为什么现代深度网络使用 ReLU 系列激活函数的根本原因。")

## 2. Sigmoid 和 Tanh（历史激活函数及其问题）

### Sigmoid
**公式**: $\sigma(x) = \frac{1}{1 + e^{-x}}$

- 输出范围 (0, 1)，历史上常用于隐藏层，现主要用于二分类输出层
- **问题1**: 梯度饱和 — 饱和区梯度趋近0，导致梯度消失
- **问题2**: 输出非零均值 — 所有输出为正，影响下一层权重更新方向
- **问题3**: 导数最大值仅0.25 — 多层叠加后梯度指数级衰减

### Tanh
**公式**: $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$

- 输出范围 (-1, 1)，零均值，比 Sigmoid 更好
- **问题**: 依然存在梯度饱和，深层网络中仍会梯度消失
- 现主要用于 RNN 隐藏层

```python
# PyTorch 用法 / PyTorch usage
nn.Sigmoid()    # Sigmoid 模块 / Sigmoid module
nn.Tanh()       # Tanh 模块 / Tanh module
torch.sigmoid(x)  # 函数式调用 / Functional call
torch.tanh(x)     # 函数式调用 / Functional call
```

In [ ]:
# Sigmoid 和 Tanh 可视化（函数 + 导数）/ Sigmoid and Tanh visualization (function + derivative)
x = np.linspace(-6, 6, 200)

# Sigmoid / Sigmoid activation
sigmoid_y = sigmoid(x)
sigmoid_dy = sigmoid_derivative(x)

# Tanh / Tanh activation
tanh_y = np.tanh(x)
tanh_dy = 1 - np.tanh(x) ** 2  # sech^2(x)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sigmoid 函数 / Sigmoid function
axes[0, 0].plot(x, sigmoid_y, 'b-', linewidth=2, label='Sigmoid')
axes[0, 0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0, 0].axhline(y=0.5, color='gray', linestyle=':', linewidth=0.5)
axes[0, 0].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[0, 0].set_title('Sigmoid: f(x) = 1/(1+e^(-x))', fontsize=12)
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('f(x)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Sigmoid 导数 / Sigmoid derivative
axes[0, 1].plot(x, sigmoid_dy, 'r-', linewidth=2, label="Sigmoid'")
axes[0, 1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0, 1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[0, 1].fill_between(x, 0, sigmoid_dy, alpha=0.15, color='red')
axes[0, 1].set_title('Sigmoid 导数（最大值仅0.25）', fontsize=12)
axes[0, 1].set_xlabel('x')
axes[0, 1].set_ylabel("f'(x)")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Tanh 函数 / Tanh function
axes[1, 0].plot(x, tanh_y, 'b-', linewidth=2, label='tanh')
axes[1, 0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1, 0].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[1, 0].set_title('Tanh: f(x) = (e^x - e^(-x))/(e^x + e^(-x))', fontsize=12)
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('f(x)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Tanh 导数 / Tanh derivative
axes[1, 1].plot(x, tanh_dy, 'r-', linewidth=2, label="tanh'")
axes[1, 1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1, 1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[1, 1].fill_between(x, 0, tanh_dy, alpha=0.15, color='red')
axes[1, 1].set_title('Tanh 导数（最大值1.0，但饱和区趋近0）', fontsize=12)
axes[1, 1].set_xlabel('x')
axes[1, 1].set_ylabel("f'(x)")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Sigmoid 与 Tanh：历史激活函数及其导数', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n结论：Sigmoid 和 Tanh 在 |x| 较大时梯度趋近0（饱和区），是梯度消失的根源。")
print("PyTorch 中：nn.Sigmoid(), nn.Tanh() 均可直接使用，但隐藏层一般不推荐。")

## 3. ReLU (Rectified Linear Unit)

**公式**: $f(x) = \max(0, x)$

**优点**:
- 计算简单高效
- 缓解梯度消失问题（正区间梯度恒为1）
- 稀疏激活（约50%神经元输出为0）

**缺点**:
- **Dead ReLU 问题**：负输入恒为0，梯度为0，神经元可能永久"死亡"
- 输出均值不为0

```python
# PyTorch 用法 / PyTorch usage
nn.ReLU()           # 作为模块层 / As a module layer
F.relu(x)           # 作为函数调用 / As a function call
```

### TF vs PyTorch 对照

| Keras | PyTorch |
|-------|---------|
| `Dense(64, activation='relu')` | `nn.Linear(784, 64)` + `nn.ReLU()` |
| `keras.layers.ReLU()` | `nn.ReLU()` |

In [ ]:
# ReLU 激活函数可视化（函数 + 导数）/ ReLU visualization (function + derivative)
x = np.linspace(-5, 5, 200)

# ReLU: max(0, x)
relu = np.maximum(0, x)
relu_grad = np.where(x > 0, 1, 0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 函数图像 / Function plot
axes[0].plot(x, relu, 'b-', linewidth=2, label='ReLU')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[0].fill_between(x[x < 0], 0, relu[x < 0], alpha=0.1, color='red', label='Dead Zone (梯度=0)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title('ReLU: f(x) = max(0, x)', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 导数图像 / Derivative plot
axes[1].plot(x, relu_grad, 'r-', linewidth=2, label="ReLU'")
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('x')
axes[1].set_ylabel("f'(x)")
axes[1].set_title('ReLU 导数', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# PyTorch 验证 / PyTorch verification
relu_module = nn.ReLU()
x_tensor = torch.tensor(x, dtype=torch.float32)
print(f"nn.ReLU 输出与 NumPy 一致: {torch.allclose(relu_module(x_tensor), torch.tensor(relu, dtype=torch.float32))}")

# PyTorch 中使用 ReLU 的三种方式 / Three ways to use ReLU in PyTorch
# 方式1: 作为 nn.Sequential 中的独立层 / As a standalone layer in Sequential
model_relu = nn.Sequential(
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

# 方式2: 在 forward() 中使用函数式 API / Functional API in forward()
# x = F.relu(x)

# 方式3: 使用 nn.ReLU 作为模块属性 / As a module attribute
# class MyModel(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.fc1 = nn.Linear(784, 64)
#         self.relu = nn.ReLU()
#         self.fc2 = nn.Linear(64, 10)
#     def forward(self, x):
#         x = self.relu(self.fc1(x))
#         return self.fc2(x)

print("\nReLU 模型构建成功")
print(model_relu)

## 4. Leaky ReLU — 解决 Dead ReLU

**公式**: $f(x) = x$ if $x > 0$, else $\alpha x$（通常 $\alpha = 0.01$ 或 $0.3$）

**优点**: 解决 Dead ReLU 问题，负区域有小梯度，神经元不会永久"死亡"

**缺点**: $\alpha$ 是超参数，需要调优

```python
# PyTorch 用法 / PyTorch usage
nn.LeakyReLU(negative_slope=0.01)   # 作为模块层 / As a module layer
F.leaky_relu(x, negative_slope=0.01) # 作为函数调用 / As a function call
```

### TF vs PyTorch 对照

| Keras | PyTorch |
|-------|---------|
| `layers.LeakyReLU(negative_slope=0.3)` | `nn.LeakyReLU(negative_slope=0.3)` |
| `activation='leaky_relu'` | `nn.LeakyReLU()` |

In [ ]:
# Leaky ReLU 可视化（函数 + 导数）/ Leaky ReLU visualization (function + derivative)
alphas = [0.01, 0.1, 0.3]
colors = ['blue', 'green', 'red']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 函数图 / Function plot
for alpha, color in zip(alphas, colors):
    leaky_relu = np.where(x > 0, x, alpha * x)
    axes[0].plot(x, leaky_relu, color=color, linewidth=2, label=f'Leaky ReLU (\u03b1={alpha})')

axes[0].plot(x, relu, 'b--', linewidth=1, alpha=0.5, label='ReLU (reference)')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title('Leaky ReLU with different \u03b1 values', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 导数图 / Derivative plot
for alpha, color in zip(alphas, colors):
    leaky_relu_grad = np.where(x > 0, 1, alpha)
    axes[1].plot(x, leaky_relu_grad, color=color, linewidth=2, label=f'Leaky ReLU (\u03b1={alpha})')

axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('x')
axes[1].set_ylabel("f'(x)")
axes[1].set_title('Leaky ReLU 导数', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# PyTorch 中使用 Leaky ReLU / Using Leaky ReLU in PyTorch
model_leaky = nn.Sequential(
    nn.Linear(784, 64),
    nn.LeakyReLU(negative_slope=0.3),
    nn.Linear(64, 10)
)

print("Leaky ReLU 模型构建成功")
print(model_leaky)

### Dead ReLU 问题演示与 Leaky ReLU 的解决方案

当所有训练样本在某个神经元的输入都为负时，ReLU 输出恒为0，梯度恒为0，权重无法更新——神经元"死亡"。
Leaky ReLU 在负区间仍有小梯度（$\alpha$），因此不会出现此问题。

In [ ]:
# Dead ReLU 演示 / Dead ReLU demonstration
torch.manual_seed(42)

# 模拟一个可能"死亡"的 ReLU 神经元 / Simulate a potentially "dead" ReLU neuron
# 权重初始化不当，使得输入全部为负 / Bad initialization making all inputs negative
dead_weight = torch.tensor([-0.8, -0.5, -0.3])
dead_bias = torch.tensor([-1.0])

# 10 个训练样本 / 10 training samples
samples = torch.randn(10, 3) * 0.5

# ReLU 前向传播 / ReLU forward pass
pre_relu = samples @ dead_weight + dead_bias
post_relu = F.relu(pre_relu)

print("=== Dead ReLU 问题演示 ===")
print(f"ReLU 前激活值: {pre_relu.detach().numpy().round(2)}")
print(f"ReLU 后输出值: {post_relu.detach().numpy().round(2)}")
print(f"所有输出为0? {torch.all(post_relu == 0).item()}")
print("梯度: 全部为0，权重无法更新!\n")

# Leaky ReLU 解决方案 / Leaky ReLU solution
post_leaky = F.leaky_relu(pre_relu, negative_slope=0.01)
print("=== Leaky ReLU 解决方案 ===")
print(f"Leaky ReLU 后输出值: {post_leaky.detach().numpy().round(4)}")
print(f"所有输出为0? {torch.all(post_leaky == 0).item()}")
print("负区间仍有小梯度 (0.01)，权重可以更新!")

# 可视化 Dead ReLU / Visualize Dead ReLU
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x_range = np.linspace(-3, 3, 200)
relu_out = np.maximum(0, x_range)
leaky_out = np.where(x_range > 0, x_range, 0.01 * x_range)

axes[0].plot(x_range, relu_out, 'b-', linewidth=2, label='ReLU')
axes[0].fill_between(x_range[x_range < 0], 0, 0, alpha=0.3, color='red', label='Dead Zone')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[0].set_title('ReLU: 负区间梯度为0（神经元死亡）', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_range, leaky_out, 'g-', linewidth=2, label='Leaky ReLU (\u03b1=0.01)')
axes[1].fill_between(x_range[x_range < 0], 0, leaky_out[x_range < 0], alpha=0.3, color='green', label='小梯度区域')
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_title('Leaky ReLU: 负区间仍有梯度', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Dead ReLU 问题与 Leaky ReLU 解决方案', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. PReLU (Parametric ReLU)

**公式**: $f(x) = x$ if $x > 0$, else $\alpha x$，其中 $\alpha$ 是**可学习参数**

**优点**: $\alpha$ 通过反向传播自动学习，无需手动调优

**缺点**: 增加模型参数量（每个通道一个 $\alpha$ 参数）

```python
# PyTorch 用法 / PyTorch usage
nn.PReLU(num_parameters=1, init=0.25)  # init 是 alpha 的初始值
# num_parameters=1: 所有通道共享一个 alpha
# num_parameters=num_channels: 每个通道一个 alpha
```

### TF vs PyTorch 对照

| Keras | PyTorch |
|-------|---------|
| `layers.PReLU()` | `nn.PReLU(num_parameters=1)` |

In [ ]:
# PReLU 可视化（函数 + 导数）/ PReLU visualization (function + derivative)
prelu_layer = nn.PReLU(num_parameters=1, init=0.25)
alpha = prelu_layer.weight.item()

prelu_y = np.where(x > 0, x, alpha * x)
prelu_dy = np.where(x > 0, 1, alpha)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 函数图 / Function plot
axes[0].plot(x, relu, 'b--', linewidth=1.5, alpha=0.5, label='ReLU')
axes[0].plot(x, prelu_y, 'g-', linewidth=2, label=f'PReLU (\u03b1={alpha:.2f}, learnable)')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[0].set_title('PReLU 激活函数', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 导数图 / Derivative plot
axes[1].plot(x, relu_grad, 'b--', linewidth=1.5, alpha=0.5, label="ReLU'")
axes[1].plot(x, prelu_dy, 'g-', linewidth=2, label=f"PReLU' (\u03b1={alpha:.2f})")
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_title('PReLU 导数', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# PyTorch 中使用 PReLU / Using PReLU in PyTorch
model_prelu = nn.Sequential(
    nn.Linear(784, 64),
    nn.PReLU(),  # \u03b1 是可学习参数 / \u03b1 is a learnable parameter
    nn.Linear(64, 32),
    nn.PReLU(),
    nn.Linear(32, 10)
)

# 查看 PReLU 的可学习参数 / View PReLU learnable parameters
print("PReLU 模型结构:")
print(model_prelu)
print()
for name, param in model_prelu.named_parameters():
    if 'weight' not in name:
        print(f"{name}: {param.data.item():.4f} (可学习参数 / learnable)")

total_params = sum(p.numel() for p in model_prelu.parameters())
print(f"\n总参数量: {total_params:,}")
print("注意: PReLU 层有可学习参数 alpha")

## 6. ELU (Exponential Linear Unit)

**公式**: $f(x) = x$ if $x > 0$, else $\alpha(e^x - 1)$

**优点**:
- 输出均值接近0（自归一化特性）
- 负区域平滑过渡，梯度不会突变
- 缓解 Dead ReLU 问题

**缺点**: 计算代价比 ReLU 高（涉及指数运算）

```python
# PyTorch 用法 / PyTorch usage
nn.ELU(alpha=1.0)     # 作为模块层 / As a module layer
F.elu(x, alpha=1.0)   # 作为函数调用 / As a function call
```

### TF vs PyTorch 对照

| Keras | PyTorch |
|-------|---------|
| `Dense(64, activation='elu')` | `nn.Linear(784, 64)` + `nn.ELU()` |
| `keras.layers.ELU(alpha=1.0)` | `nn.ELU(alpha=1.0)` |

In [ ]:
# ELU 可视化（函数 + 导数）/ ELU visualization (function + derivative)
alpha = 1.0
elu = np.where(x > 0, x, alpha * (np.exp(x) - 1))
elu_grad = np.where(x > 0, 1, alpha * np.exp(x))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 函数对比 / Function comparison
axes[0].plot(x, relu, 'b--', linewidth=2, label='ReLU', alpha=0.7)
axes[0].plot(x, elu, 'r-', linewidth=2, label=f'ELU (\u03b1={alpha})')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].axhline(y=-alpha, color='gray', linestyle=':', linewidth=0.5, label=f'渐近线 y = -{alpha}')
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title('ELU vs ReLU', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-2, 5)

# 导数对比 / Derivative comparison
axes[1].plot(x, relu_grad, 'b--', linewidth=2, label="ReLU'", alpha=0.7)
axes[1].plot(x, elu_grad, 'r-', linewidth=2, label="ELU'")
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('x')
axes[1].set_ylabel("f'(x)")
axes[1].set_title('ELU vs ReLU 导数', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# PyTorch 验证 / PyTorch verification
elu_module = nn.ELU(alpha=1.0)
x_t = torch.tensor(x, dtype=torch.float32)
print(f"nn.ELU 输出与 NumPy 一致: {torch.allclose(elu_module(x_t), torch.tensor(elu, dtype=torch.float32), atol=1e-5)}")

# PyTorch 中使用 ELU / Using ELU in PyTorch
model_elu = nn.Sequential(
    nn.Linear(784, 64),
    nn.ELU(alpha=1.0),
    nn.Linear(64, 32),
    nn.ELU(alpha=1.0),
    nn.Linear(32, 10)
)

print("ELU 模型构建成功")
print(model_elu)

## 7. SELU (Scaled ELU)

**公式**: $f(x) = \lambda \times (x$ if $x > 0$, else $\alpha(e^x - 1))$

其中 $\lambda \approx 1.0507$, $\alpha \approx 1.6733$ 是精心选择的常数。

**优点**: 自归一化特性 — 在特定条件下，网络输出自动保持均值为0、方差为1

**使用条件**:
- 必须使用 LeCun 初始化（`nn.init.lecun_normal_` 或 `kaiming_normal_` with `nonlinearity='linear'`）
- 必须使用 `nn.AlphaDropout`（而非普通 Dropout）
- 网络必须是全连接的（Sequential Dense layers）

```python
# PyTorch 用法 / PyTorch usage
nn.SELU()     # 作为模块层 / As a module layer
F.selu(x)     # 作为函数调用 / As a function call
```

### TF vs PyTorch 对照

| Keras | PyTorch |
|-------|---------|
| `Dense(64, activation='selu', kernel_initializer='lecun_normal')` | `nn.Linear(784, 64)` + `nn.SELU()` + LeCun init |
| `layers.AlphaDropout(0.1)` | `nn.AlphaDropout(0.1)` |

In [ ]:
# SELU 可视化（函数 + 导数）/ SELU visualization (function + derivative)
lambda_selu = 1.0507
alpha_selu = 1.6733

selu = lambda_selu * np.where(x > 0, x, alpha_selu * (np.exp(x) - 1))
selu_grad = lambda_selu * np.where(x > 0, 1, alpha_selu * np.exp(x))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 函数对比 / Function comparison
axes[0].plot(x, relu, 'b--', linewidth=1.5, label='ReLU', alpha=0.5)
axes[0].plot(x, elu, 'g--', linewidth=1.5, label='ELU', alpha=0.5)
axes[0].plot(x, selu, 'r-', linewidth=2, label='SELU')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title('SELU vs ELU vs ReLU', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-2, 5)

# 导数对比 / Derivative comparison
axes[1].plot(x, relu_grad, 'b--', linewidth=1.5, label="ReLU'", alpha=0.5)
axes[1].plot(x, elu_grad, 'g--', linewidth=1.5, label="ELU'", alpha=0.5)
axes[1].plot(x, selu_grad, 'r-', linewidth=2, label="SELU'")
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('x')
axes[1].set_ylabel("f'(x)")
axes[1].set_title('SELU vs ELU vs ReLU 导数', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# LeCun 初始化函数 / LeCun initialization function
def init_lecun(module):
    """
    LeCun 初始化（用于 SELU）
    LeCun initialization for SELU activation.

    Parameters / 参数:
    -----------
    module : nn.Module
        模型模块 / Model module
    """
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight, nonlinearity='linear')
        if module.bias is not None:
            nn.init.zeros_(module.bias)

# SELU 自归一化网络示例 / SELU self-normalizing network example
model_selu = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 256),
    nn.SELU(),
    nn.AlphaDropout(0.1),  # SELU 专用 Dropout
    nn.Linear(256, 128),
    nn.SELU(),
    nn.AlphaDropout(0.1),
    nn.Linear(128, 64),
    nn.SELU(),
    nn.Linear(64, 10)
)

# 应用 LeCun 初始化 / Apply LeCun initialization
model_selu.apply(init_lecun)

total_params = sum(p.numel() for p in model_selu.parameters())
print(f"SELU 自归一化网络参数量: {total_params:,}")
print(f"PyTorch 中 nn.AlphaDropout 可用: {hasattr(nn, 'AlphaDropout')}")
print("\nSELU 自归一化网络构建成功")

## 8. 新一代激活函数：GELU 和 SiLU (Swish)

### GELU (Gaussian Error Linear Unit)
**公式**: $\text{GELU}(x) = x \cdot \Phi(x)$，其中 $\Phi(x)$ 是标准正态分布的累积分布函数

- 被 Transformer 架构广泛采用（BERT, GPT 等）
- 在 x=0 附近平滑过渡，不像 ReLU 那样硬截断

### SiLU (Swish)
**公式**: $\text{SiLU}(x) = x \cdot \sigma(x)$

- 由 Google Brain 提出，在深层网络中表现优于 ReLU
- 非单调性，在负区间有微小下凹

```python
# PyTorch 用法 / PyTorch usage
nn.GELU()    # GELU 激活 / GELU activation
F.gelu(x)    # GELU 函数 / GELU function
nn.SiLU()    # SiLU 激活 (PyTorch >= 1.7) / SiLU activation
F.silu(x)    # SiLU 函数 / SiLU function
```

### TF vs PyTorch 对照

| Keras / TF | PyTorch |
|------------|---------|
| `tf.nn.gelu` | `nn.GELU()` |
| `tf.nn.swish` | `nn.SiLU()` (PyTorch 1.7+) |

In [ ]:
# GELU 和 SiLU 可视化（函数 + 导数）/ GELU and SiLU visualization (function + derivative)
from scipy.stats import norm

# GELU: x * Phi(x) / GELU function
gelu = x * norm.cdf(x)
gelu_grad = norm.cdf(x) + x * norm.pdf(x)

# SiLU (Swish): x * sigmoid(x) / SiLU function
silu = x * sigmoid(x)
silu_dy = sigmoid(x) + x * sigmoid(x) * (1 - sigmoid(x))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 函数对比 / Function comparison
axes[0].plot(x, relu, 'b--', linewidth=1.5, label='ReLU', alpha=0.5)
axes[0].plot(x, gelu, 'r-', linewidth=2, label='GELU')
axes[0].plot(x, silu, 'g-', linewidth=2, label='SiLU (Swish)')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title('GELU & SiLU vs ReLU', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 导数对比 / Derivative comparison
axes[1].plot(x, relu_grad, 'b--', linewidth=1.5, label="ReLU'", alpha=0.5)
axes[1].plot(x, gelu_grad, 'r-', linewidth=2, label="GELU'")
axes[1].plot(x, silu_dy, 'g-', linewidth=2, label="SiLU'")
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('x')
axes[1].set_ylabel("f'(x)")
axes[1].set_title('GELU & SiLU 导数', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# PyTorch 验证 / PyTorch verification
gelu_module = nn.GELU()
silu_module = nn.SiLU()
x_t = torch.tensor(x, dtype=torch.float32)
print("nn.GELU 可用: \u2713")
print("nn.SiLU 可用: \u2713")
print(f"GELU 输出范围: [{gelu_module(x_t).min().item():.3f}, {gelu_module(x_t).max().item():.3f}]")
print(f"SiLU 输出范围: [{silu_module(x_t).min().item():.3f}, {silu_module(x_t).max().item():.3f}]")

## 9. 所有激活函数对比可视化

In [ ]:
# 所有激活函数对比（函数 + 导数）/ All activation functions comparison (function + derivative)
x = np.linspace(-4, 4, 300)

activations = {
    'Sigmoid': sigmoid(x),
    'Tanh': np.tanh(x),
    'ReLU': np.maximum(0, x),
    'Leaky ReLU (\u03b1=0.1)': np.where(x > 0, x, 0.1 * x),
    'PReLU (\u03b1=0.25)': np.where(x > 0, x, 0.25 * x),
    'ELU (\u03b1=1)': np.where(x > 0, x, 1.0 * (np.exp(x) - 1)),
    'SELU': 1.0507 * np.where(x > 0, x, 1.6733 * (np.exp(x) - 1)),
    'GELU': x * norm.cdf(x),
    'SiLU (Swish)': x * sigmoid(x),
}

derivatives = {
    'Sigmoid': sigmoid(x) * (1 - sigmoid(x)),
    'Tanh': 1 - np.tanh(x) ** 2,
    'ReLU': np.where(x > 0, 1, 0),
    'Leaky ReLU (\u03b1=0.1)': np.where(x > 0, 1, 0.1),
    'PReLU (\u03b1=0.25)': np.where(x > 0, 1, 0.25),
    'ELU (\u03b1=1)': np.where(x > 0, 1, 1.0 * np.exp(x)),
    'SELU': 1.0507 * np.where(x > 0, 1, 1.6733 * np.exp(x)),
    'GELU': norm.cdf(x) + x * norm.pdf(x),
    'SiLU (Swish)': sigmoid(x) + x * sigmoid(x) * (1 - sigmoid(x)),
}

# 绘制函数 / Plot functions
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes_flat = axes.flatten()

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
          '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']

for idx, (name, values) in enumerate(activations.items()):
    ax = axes_flat[idx]
    ax.plot(x, values, color=colors[idx], linewidth=2, label=name)
    # 绘制导数（虚线）/ Plot derivative (dashed)
    ax.plot(x, derivatives[name], color=colors[idx], linewidth=1, linestyle='--', alpha=0.6, label=f"{name} '")
    ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('f(x) / f\'(x)')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-2, 4)

plt.suptitle('深度学习常用激活函数及其导数对比\n(实线=函数, 虚线=导数)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('activation_functions_comparison_pytorch.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. 实际应用：Fashion-MNIST 训练对比

使用不同激活函数在 Fashion-MNIST 数据集上训练，对比训练速度和最终精度。

In [ ]:
# 加载 Fashion-MNIST 数据集 / Load Fashion-MNIST dataset
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"训练集大小: {len(train_dataset)}")
print(f"测试集大小: {len(test_dataset)}")

In [ ]:
def init_he(module):
    """
    He 初始化（用于 ReLU 系列激活函数）
    He initialization for ReLU family activation functions.

    Parameters / 参数:
    -----------
    module : nn.Module
        模型模块 / Model module
    """
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
        if module.bias is not None:
            nn.init.zeros_(module.bias)


def build_model_with_activation(activation_name):
    """
    构建使用指定激活函数的 PyTorch 模型
    Build a PyTorch model with the specified activation function.

    Parameters / 参数:
    -----------
    activation_name : str
        激活函数名称 / Activation function name:
        'relu', 'leaky_relu', 'prelu', 'elu', 'selu', 'gelu', 'silu'

    Returns / 返回:
    --------
    nn.Sequential
        构建好的模型 / Built model
    """
    layers_list = [nn.Flatten()]
    in_features = 784
    hidden_sizes = [256, 128, 64]

    for units in hidden_sizes:
        linear = nn.Linear(in_features, units)
        layers_list.append(linear)

        if activation_name == 'relu':
            layers_list.append(nn.ReLU())
        elif activation_name == 'leaky_relu':
            layers_list.append(nn.LeakyReLU(negative_slope=0.2))
        elif activation_name == 'prelu':
            layers_list.append(nn.PReLU(init=0.25))
        elif activation_name == 'elu':
            layers_list.append(nn.ELU())
        elif activation_name == 'selu':
            layers_list.append(nn.SELU())
            layers_list.append(nn.AlphaDropout(0.1))
        elif activation_name == 'gelu':
            layers_list.append(nn.GELU())
        elif activation_name == 'silu':
            layers_list.append(nn.SiLU())

        in_features = units

    layers_list.append(nn.Linear(in_features, 10))
    model = nn.Sequential(*layers_list)

    # 根据激活函数选择初始化方法 / Choose initialization based on activation
    if activation_name == 'selu':
        model.apply(init_lecun)
    else:
        model.apply(init_he)

    return model


# 测试各种激活函数的模型构建 / Test model building with different activations
activation_list = ['relu', 'leaky_relu', 'prelu', 'elu', 'selu', 'gelu', 'silu']

for act in activation_list:
    model = build_model_with_activation(act)
    param_count = sum(p.numel() for p in model.parameters())
    print(f"{act.upper():12s} - 参数量: {param_count:,}")

In [ ]:
def train_and_evaluate(model, train_loader, test_loader, epochs=10, lr=1e-3):
    """
    训练并评估模型
    Train and evaluate a model.

    Parameters / 参数:
    -----------
    model : nn.Module
        待训练的模型 / Model to train
    train_loader : DataLoader
        训练数据加载器 / Training data loader
    test_loader : DataLoader
        测试数据加载器 / Test data loader
    epochs : int
        训练轮数 / Number of epochs
    lr : float
        学习率 / Learning rate

    Returns / 返回:
    --------
    dict : 训练历史 / Training history
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(epochs):
        # 训练阶段 / Training phase
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(y_batch).sum().item()
            total += y_batch.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # 测试阶段 / Test phase
        model.eval()
        test_correct, test_total = 0, 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                _, predicted = outputs.max(1)
                test_correct += predicted.eq(y_batch).sum().item()
                test_total += y_batch.size(0)

        test_acc = test_correct / test_total

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:2d}/{epochs} - "
                  f"Loss: {train_loss:.4f}, "
                  f"Train Acc: {train_acc:.4f}, "
                  f"Test Acc: {test_acc:.4f}")

    return history

In [ ]:
# 训练不同激活函数的模型并对比 / Train models with different activations and compare
EPOCHS = 10
results = {}

for act_name in activation_list:
    print(f"\n{'='*50}")
    print(f"训练激活函数: {act_name.upper()}")
    print(f"{'='*50}")

    torch.manual_seed(RANDOM_SEED)
    model = build_model_with_activation(act_name)
    param_count = sum(p.numel() for p in model.parameters())
    print(f"参数量: {param_count:,}")

    history = train_and_evaluate(model, train_loader, test_loader, epochs=EPOCHS)
    results[act_name] = history

print("\n" + "="*50)
print("所有模型训练完成！")
print("="*50)

In [ ]:
# 可视化训练结果对比 / Visualize training results comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
          '#8c564b', '#e377c2']

for idx, (act_name, history) in enumerate(results.items()):
    epochs_range = range(1, EPOCHS + 1)
    axes[0].plot(epochs_range, history['train_loss'], color=colors[idx],
                 linewidth=1.5, label=act_name.upper())
    axes[1].plot(epochs_range, history['train_acc'], color=colors[idx],
                 linewidth=1.5, label=act_name.upper())
    axes[2].plot(epochs_range, history['test_acc'], color=colors[idx],
                 linewidth=1.5, label=act_name.upper())

axes[0].set_title('Training Loss', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Training Accuracy', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

axes[2].set_title('Test Accuracy', fontsize=13)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Fashion-MNIST: 不同激活函数训练对比', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 打印最终结果 / Print final results
print("\n最终测试精度对比 / Final Test Accuracy Comparison:")
print("-" * 45)
for act_name, history in sorted(results.items(), key=lambda x: x[1]['test_acc'][-1], reverse=True):
    print(f"  {act_name.upper():12s}: {history['test_acc'][-1]:.4f}")

## 11. 输出层激活函数：Softmax 与 Sigmoid

### Softmax（多分类）

**公式**: $f(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$

**重要**: 在 PyTorch 中，`nn.CrossEntropyLoss` 已经内置了 Softmax 操作，
因此模型输出层不需要添加 `nn.Softmax()`，直接输出原始 logits 即可。
这与 Keras 中 `Dense(10, activation='softmax')` + `sparse_categorical_crossentropy` 不同。

### Sigmoid（二分类）

**公式**: $f(x) = \frac{1}{1 + e^{-x}}$

同样，`nn.BCEWithLogitsLoss` 内置了 Sigmoid 操作。

### TF vs PyTorch 对照

| Keras | PyTorch |
|-------|---------|
| `Dense(10, activation='softmax')` + `sparse_categorical_crossentropy` | `nn.Linear(64, 10)` + `nn.CrossEntropyLoss()` (无需 Softmax) |
| `Dense(1, activation='sigmoid')` + `binary_crossentropy` | `nn.Linear(64, 1)` + `nn.BCEWithLogitsLoss()` (无需 Sigmoid) |
| 如需概率输出: `model.predict(X)` | 如需概率输出: `torch.softmax(logits, dim=1)` |

In [ ]:
# 输出层激活函数示例 / Output layer activation examples

# --- 多分类示例 / Multi-class example ---
# Keras: Dense(10, activation='softmax') + sparse_categorical_crossentropy
# PyTorch: Linear(64, 10) + CrossEntropyLoss (不需要 Softmax 层)
model_multiclass = nn.Sequential(
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 10)  # 输出原始 logits，不添加 Softmax
)
criterion_multi = nn.CrossEntropyLoss()  # 内置 Softmax

# 如果需要获取概率分布 / If probability distribution is needed
logits = torch.randn(2, 10)
probs = torch.softmax(logits, dim=1)
print("多分类概率分布:")
print(probs)
print(f"概率之和: {probs.sum(dim=1).item():.6f}")

# --- 二分类示例 / Binary classification example ---
# Keras: Dense(1, activation='sigmoid') + binary_crossentropy
# PyTorch: Linear(64, 1) + BCEWithLogitsLoss (不需要 Sigmoid)
model_binary = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.Linear(64, 1)  # 输出原始 logits，不添加 Sigmoid
)
criterion_binary = nn.BCEWithLogitsLoss()  # 内置 Sigmoid

# 如果需要获取概率 / If probability is needed
logits_binary = torch.randn(3, 1)
probs_binary = torch.sigmoid(logits_binary)
print(f"\n二分类概率: {probs_binary.flatten().tolist()}")

print("\n注意: 训练时不要使用 nn.Softmax + nn.CrossEntropyLoss，会导致双重 Softmax")
print("正确做法: 训练用 nn.CrossEntropyLoss，推理时用 torch.softmax() 获取概率")

## 12. 激活函数选择指南

### 推荐策略

| 场景 | 推荐激活函数 | PyTorch 代码 | 配套初始化 |
|------|--------------|-------------|-----------|
| 默认选择 | ReLU | `nn.ReLU()` | He |
| 担心 Dead ReLU | Leaky ReLU / ELU | `nn.LeakyReLU(0.2)` / `nn.ELU()` | He |
| 自归一化网络 | SELU | `nn.SELU()` | LeCun |
| Transformer | GELU | `nn.GELU()` | He |
| 现代高效网络 | SiLU (Swish) | `nn.SiLU()` | He |
| 二分类输出层 | sigmoid | `torch.sigmoid()` | - |
| 多分类输出层 | softmax | `torch.softmax()` / `CrossEntropyLoss` | - |
| RNN 隐藏层 | tanh | `nn.Tanh()` | Glorot |

### 性能对比（一般规律）

- **训练速度**: ReLU > Leaky ReLU > ELU > SELU > GELU > SiLU
- **模型性能**: 通常差异不大，GELU/SiLU 在 Transformer 中有优势，ELU/SELU 在深层全连接网络中略有优势
- **稳定性**: SELU（满足条件时）> ELU > Leaky ReLU > ReLU

### PyTorch 激活函数速查表

| 激活函数 | 类 | 函数式 API |
|----------|-----|-----------|
| ReLU | `nn.ReLU()` | `F.relu()` |
| Leaky ReLU | `nn.LeakyReLU(negative_slope)` | `F.leaky_relu()` |
| PReLU | `nn.PReLU()` | - |
| ELU | `nn.ELU(alpha)` | `F.elu()` |
| SELU | `nn.SELU()` | `F.selu()` |
| GELU | `nn.GELU()` | `F.gelu()` |
| SiLU/Swish | `nn.SiLU()` | `F.silu()` |
| Tanh | `nn.Tanh()` | `torch.tanh()` |
| Sigmoid | `nn.Sigmoid()` | `torch.sigmoid()` |
| Softmax | `nn.Softmax(dim)` | `torch.softmax()` |

## 13. TF vs PyTorch 对照

### 激活函数 API 对照表

| 功能 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| ReLU | `layers.ReLU()` 或 `activation='relu'` | `nn.ReLU()` 或 `F.relu(x)` |
| Leaky ReLU | `layers.LeakyReLU(negative_slope=0.3)` | `nn.LeakyReLU(negative_slope=0.3)` 或 `F.leaky_relu(x, 0.3)` |
| PReLU | `layers.PReLU()` | `nn.PReLU(num_parameters=1, init=0.25)` |
| ELU | `layers.ELU()` 或 `activation='elu'` | `nn.ELU(alpha=1.0)` 或 `F.elu(x)` |
| SELU | `activation='selu'` | `nn.SELU()` 或 `F.selu(x)` |
| GELU | `activation='gelu'` (TF 2.4+) | `nn.GELU()` 或 `F.gelu(x)` |
| SiLU/Swish | `activation='swish'` | `nn.SiLU()` 或 `F.silu(x)` |
| Sigmoid | `activation='sigmoid'` | `nn.Sigmoid()` 或 `torch.sigmoid(x)` |
| Tanh | `activation='tanh'` | `nn.Tanh()` 或 `torch.tanh(x)` |
| Softmax | `activation='softmax'` | `nn.Softmax(dim=-1)` 或 `F.softmax(x, dim=-1)` |

### 关键差异

| 差异点 | TensorFlow / Keras | PyTorch |
|--------|-------------------|---------|
| 激活函数使用方式 | 可作为 Dense 层参数 `activation='relu'` | 需作为独立层 `nn.ReLU()` 或函数 `F.relu(x)` |
| 权重初始化 | 字符串指定 `kernel_initializer='he_normal'` | 函数调用 `nn.init.kaiming_normal_(weight)` |
| AlphaDropout | `layers.AlphaDropout(0.1)` | `nn.AlphaDropout(p=0.1)` |
| PReLU 参数 | 共享参数（默认） | `num_parameters` 控制是否逐通道 |
| Softmax 输出 | `Dense(10, activation='softmax')` | `nn.Linear(64, 10)` + `nn.CrossEntropyLoss()` |
| Sigmoid 输出 | `Dense(1, activation='sigmoid')` | `nn.Linear(64, 1)` + `nn.BCEWithLogitsLoss()` |
| 函数式调用 | `tf.nn.relu(x)` | `F.relu(x)` 或 `torch.relu(x)` |
| 激活+初始化 | `Dense(64, activation='selu', kernel_initializer='lecun_normal')` | `nn.Linear(784, 64)` + `nn.SELU()` + `model.apply(init_lecun)` |

In [ ]:
# TF vs PyTorch 对照代码示例 / TF vs PyTorch comparison code examples

# === TensorFlow / Keras 写法 ===
# model_tf = keras.Sequential([
#     layers.Dense(256, activation='relu', kernel_initializer='he_normal'),
#     layers.Dense(128, kernel_initializer='he_normal'),
#     layers.LeakyReLU(negative_slope=0.3),
#     layers.Dense(64, kernel_initializer='he_normal'),
#     layers.PReLU(),
#     layers.Dense(10, activation='softmax')
# ])

# === PyTorch 等效写法 ===
model_pt = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.LeakyReLU(negative_slope=0.3),
    nn.Linear(128, 64),
    nn.PReLU(),
    nn.Linear(64, 10),
)

# He 初始化 / He initialization
for layer in model_pt:
    if isinstance(layer, nn.Linear):
        nn.init.kaiming_normal_(layer.weight)
        nn.init.zeros_(layer.bias)

print("PyTorch 模型结构:")
print(model_pt)
print(f"\n参数量: {sum(p.numel() for p in model_pt.parameters()):,}")

# 函数式调用对比 / Functional call comparison
x_demo = torch.randn(2, 784)
print(f"\nF.relu(x)    = nn.ReLU()(x)    : {torch.allclose(F.relu(x_demo), nn.ReLU()(x_demo))}")
print(f"F.elu(x)     = nn.ELU()(x)     : {torch.allclose(F.elu(x_demo), nn.ELU()(x_demo))}")
print(f"F.selu(x)    = nn.SELU()(x)    : {torch.allclose(F.selu(x_demo), nn.SELU()(x_demo))}")
print(f"F.gelu(x)    = nn.GELU()(x)    : {torch.allclose(F.gelu(x_demo), nn.GELU()(x_demo))}")
print(f"F.silu(x)    = nn.SiLU()(x)    : {torch.allclose(F.silu(x_demo), nn.SiLU()(x_demo))}")
print(f"F.leaky_relu = nn.LeakyReLU() : {torch.allclose(F.leaky_relu(x_demo), nn.LeakyReLU()(x_demo))}")

## 14. 练习

### 练习 1：观察 Dying ReLU 现象

构建一个 10 层深层网络，使用 ReLU 激活函数，训练 20 个 epoch。统计每层中输出为 0 的神经元比例，观察是否有大量神经元"死亡"。

**提示**:
- 在 `forward()` 中记录每层 ReLU 后为 0 的比例
- 使用较大的学习率（如 0.1）更容易触发 Dying ReLU
- 对比使用 Leaky ReLU 时的零输出比例

```python
# 参考框架 / Reference framework
class DeepReLUNet(nn.Module):
    def __init__(self, activation='relu', num_layers=10):
        super().__init__()
        # TODO: 构建多层网络，记录每层激活统计
        pass

    def forward(self, x):
        # TODO: 前向传播，记录每层零输出比例
        pass
```

---

### 练习 2：实现自定义激活函数 Mish

Mish 激活函数的公式为：$\text{Mish}(x) = x \cdot \tanh(\ln(1 + e^x))$

要求：
1. 创建 `nn.Module` 子类实现 Mish
2. 绘制 Mish 与 ReLU、SiLU 的函数和导数对比图
3. 在 Fashion-MNIST 上对比 Mish 与 ReLU 的训练效果

**提示**:
- PyTorch 中可用 `torch.log1p(torch.exp(x))` 代替 `ln(1+e^x)` 以提高数值稳定性
- `nn.Module` 子类只需实现 `forward()` 方法

```python
# 参考框架 / Reference framework
class Mish(nn.Module):
    def forward(self, x):
        # TODO: 实现 Mish 激活函数
        pass
```

---

### 练习 3：SELU 自归一化特性验证

验证 SELU 的自归一化特性：在深层网络中，逐层统计输出的均值和方差，对比使用 ReLU 和 SELU 时各层输出分布的变化。

**提示**:
- 生成标准正态分布的输入数据
- 构建一个 20 层的全连接网络，分别使用 ReLU + He 初始化 和 SELU + LeCun 初始化
- 逐层记录输出的均值和方差
- 绘制"层数 vs 均值/方差"的折线图

```python
# 参考框架 / Reference framework
def check_self_normalizing(activation='selu', num_layers=20):
    # TODO: 构建网络，逐层统计均值和方差
    pass
```

In [ ]:
# 练习区域 / Exercise area
# 请在上方练习描述的指导下完成代码 / Complete the code guided by the exercise descriptions above

print("激活函数模块测试完成！")
print("\n关键要点 / Key Takeaways:")
print("1. ReLU 是默认首选，简单高效 / ReLU is the default choice, simple and efficient")
print("2. LeakyReLU/PReLU/ELU 解决 Dead ReLU 问题 / Solve Dead ReLU problem")
print("3. SELU 实现自归一化，需要特定配置 / SELU enables self-normalization with specific config")
print("4. GELU/SiLU 是新一代激活函数，被 Transformer 采用 / GELU/SiLU are next-gen activations")
print("5. 输出层使用 sigmoid(二分类) 或 softmax(多分类) / Use sigmoid/softmax for output layer")
print("6. PyTorch 中 nn.ReLU/F.relu 两种调用方式等效 / nn.ReLU and F.relu are equivalent")